# RFM Customer Segmentation

Segment customers by Recency, Frequency, Monetary value.

**RFM Metrics:**
- **Recency (R)**: Days since last purchase (lower = more recent = better)
- **Frequency (F)**: Number of purchases (higher = more loyal = better)
- **Monetary (M)**: Total spent (higher = more valuable = better)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Synthetic RFM data
np.random.seed(42)
n_customers = 1000

data = {
    'customer_id': range(1, n_customers + 1),
    'recency': np.random.exponential(30, n_customers),  # Days since purchase
    'frequency': np.random.poisson(5, n_customers),  # Number of purchases
    'monetary': np.random.exponential(100, n_customers)  # Total spent
}

df = pd.DataFrame(data)

# RFM Scores (1-5, higher is better)
df['r_score'] = pd.qcut(df['recency'], 5, labels=[5, 4, 3, 2, 1], duplicates='drop')
df['f_score'] = pd.qcut(df['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5], duplicates='drop')
df['m_score'] = pd.qcut(df['monetary'], 5, labels=[1, 2, 3, 4, 5], duplicates='drop')

print('RFM Statistics:')
print(df[['recency', 'frequency', 'monetary']].describe().round(2))
print()

# Segmentation
def assign_segment(row):
    r, f, m = int(row['r_score']), int(row['f_score']), int(row['m_score'])
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif f >= 4 and m >= 3:
        return 'Loyal'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r >= 3 and f <= 2:
        return 'Potential'
    elif r == 1:
        return 'Lost'
    else:
        return 'Other'

df['segment'] = df.apply(assign_segment, axis=1)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Segment distribution
segment_counts = df['segment'].value_counts()
axes[0, 0].barh(segment_counts.index, segment_counts.values, color='steelblue')
axes[0, 0].set_xlabel('Customer Count')
axes[0, 0].set_title('Customers by Segment')

# Monetary by segment
df.boxplot(column='monetary', by='segment', ax=axes[0, 1])
axes[0, 1].set_title('Monetary Value by Segment')
axes[0, 1].set_xlabel('Segment')

# Recency vs Frequency scatter
colors = {'Champions': 'gold', 'Loyal': 'green', 'At Risk': 'orange', 'Potential': 'blue', 'Lost': 'red', 'Other': 'gray'}
for segment in df['segment'].unique():
    mask = df['segment'] == segment
    axes[1, 0].scatter(df[mask]['recency'], df[mask]['frequency'], 
                      label=segment, alpha=0.6, color=colors.get(segment, 'gray'))
axes[1, 0].set_xlabel('Recency (days since purchase)')
axes[1, 0].set_ylabel('Frequency (# purchases)')
axes[1, 0].set_title('Recency vs Frequency')
axes[1, 0].legend()

# Segment summary
summary = df.groupby('segment').agg({
    'customer_id': 'count',
    'monetary': 'mean',
    'frequency': 'mean',
    'recency': 'mean'
}).round(2)
axes[1, 1].axis('off')
axes[1, 1].table(cellText=summary.values, colLabels=summary.columns, 
                 rowLabels=summary.index, loc='center')
axes[1, 1].set_title('Segment Summary')

plt.tight_layout()
plt.show()

print('\nSegment Summary:')
print(summary)